In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, mean_squared_error

import torch.nn as nn
import torch.optim as optim
import math
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader
import datetime

import random
from skopt import BayesSearchCV
from sklearn.metrics import r2_score
from sklearn.model_selection import PredefinedSplit

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
device

device(type='cuda')

In [6]:
data = pd.read_excel('../Data/New_data11.xlsx')
data_pop = pd.read_excel('../Data/population_2013_2072.xlsx')

In [7]:
for i in range(2013,2025):
    data_pop.loc[data_pop['year']==i,0] = data.loc[data['Year']==i,'Population_0'].iloc[0]
    data_pop.loc[data_pop['year']==i,40] = data.loc[data['Year']==i,'Population_40'].iloc[0]
    data_pop.loc[data_pop['year']==i,60] = data.loc[data['Year']==i,'Population_60'].iloc[0]

In [8]:
Climber = pd.read_excel('../Data/Climber.xlsx')

In [9]:
chuseok_dates = pd.read_excel('../Data/추석날짜.xlsx')

In [10]:
chuseok_week = []
# 각 해의 추석 날짜가 몇 번째 주에 속하는지 계산
for date_str in chuseok_dates['date'].values:
    chuseok_date = date_str.astype('datetime64[s]').astype(datetime.datetime)
    chuseok_week.append(chuseok_date.isocalendar()[1])  # ISO 캘린더에서 주차 계산
print(chuseok_week)

[37, 39, 37, 40, 39, 37, 40, 38, 36, 39, 38, 41, 39, 37, 40, 38, 37, 40, 38, 36, 39, 37, 40, 39, 37, 39, 38, 37, 39, 38, 40, 39, 37, 40, 39, 36, 39, 38, 36, 39, 38, 40, 38, 37, 40, 38, 37, 39, 37, 40, 39, 38, 39, 38, 37, 39, 38]


In [11]:
data['Chuseok'] = 0

In [12]:
for j, i in enumerate(range(2014,2025)):
    data.loc[(data['Year']==i) & (data['Week']>=chuseok_week[j]-2) & (data['Week']<=chuseok_week[j]+2), 'Chuseok']=1

In [13]:
data['Elder_cases'] = data['Cases_60']
data['Elder_incidence'] = 0.0

data['Nonelder_cases'] = data['Cases_0']+data['Cases_40']
data['Nonelder_incidence'] = 0.0

In [14]:
for i in range(2013,2025):
    data.loc[data['Year']==i,'Elder_incidence'] = 1000000*data.loc[data['Year']==i,'Elder_cases']/data_pop.loc[data_pop['year']==i,60].values[0]
    data.loc[data['Year']==i,'Nonelder_incidence'] = 1000000*data.loc[data['Year']==i,'Nonelder_cases']/(data_pop.loc[data_pop['year']==i,40].values[0] + data_pop.loc[data_pop['year']==i,0].values[0])

In [15]:
observed_year_incidence = pd.DataFrame(columns=['year','Total incidence','Elder incidence','Nonelder incidence'])
observed_year_cases = pd.DataFrame(columns=['year','Total cases','Elder cases','Nonelder cases'])
for num, i in enumerate(range(2015,2025)):
    observed_year_incidence.loc[num,'year'] = i
    observed_year_cases.loc[num,'year'] = i
    observed_year_cases.loc[num,'Total cases'] = data.loc[(data['Year']==i),'Cases'].sum()
    observed_year_incidence.loc[num,'Total incidence'] = 1000000*observed_year_cases.loc[num,'Total cases']/data_pop.loc[data_pop['year']==i,[0,40,60]].sum(axis=1).values[0]
    observed_year_incidence.loc[num,'Elder incidence'] = data.loc[(data['Year']==i),'Elder_incidence'].sum()
    observed_year_incidence.loc[num,'Nonelder incidence'] = data.loc[(data['Year']==i),'Nonelder_incidence'].sum()
    observed_year_cases.loc[num,'Elder cases'] = data.loc[(data['Year']==i),'Elder_cases'].sum()
    observed_year_cases.loc[num,'Nonelder cases'] = data.loc[(data['Year']==i),'Nonelder_cases'].sum()

In [16]:
observed_year_incidence['rate']=observed_year_incidence['Elder incidence']/observed_year_incidence['Nonelder incidence']

In [17]:
observed_year_incidence

,year,Total incidence,Elder incidence,Nonelder incidence,rate
0,2015,1.548566,5.811451,0.599191,9.698835
1,2016,3.221536,11.485501,1.278144,8.986078
2,2017,5.295753,18.365782,2.042529,8.991686
3,2018,5.020834,17.0151,1.861381,9.141117
4,2019,4.307945,14.668272,1.409221,10.408781
5,2020,4.68784,14.540912,1.730624,8.402119
6,2021,3.322417,10.368342,1.047658,9.89669
7,2022,3.735057,11.424875,1.115628,10.24076
8,2023,3.828853,11.82998,0.946905,12.493313
9,2024,3.30428,10.106488,0.744646,13.572198


In [18]:
data['Weekly hiker'] = data['Weekly hiker']*(data['Population_60']/data['Population'])

In [19]:
start_year = 2015
end_year = 2023

data_train = data[(data['Year']>=start_year) & (data['Year']<end_year)]
data_test = data[data['Year']>=end_year]

In [20]:
features = ['tem','rain', 'hum', 'Chuseok', 'Weekly hiker', 'Tick Density']

In [21]:
target = ['Elder_incidence']

In [22]:
def make_dataset_D(x_data, y_data, window_size):
    x_list = []
    y_list = []
    for i in range(len(x_data) - window_size+1):
        x_list.append(np.array(x_data.iloc[i:i+window_size]))
        y_list.append(np.array(y_data.iloc[i+window_size-1]))
    x_list = np.array(x_list)
    y_list = np.array(y_list).reshape(-1)
    return x_list, y_list

In [23]:
import random
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [24]:
def block_bootstrap_indices(n, block_len, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    idx = []
    while len(idx) < n:
        start = rng.integers(0, n - block_len + 1)
        idx.extend(range(start, start + block_len))
    return np.array(idx[:n])

In [25]:
test_num = 105
valid_num = 52

seed_value = 42

In [26]:
window_size_LR = 6
machine_values = ['LR','RF','XGB','DNN','LSTM']

In [27]:
data_temp = data_train[(data_train['Year']>start_year) | (data_train['Week']>53-window_size_LR+1)]
data_temp.index = range(len(data_temp))
data_test.index = range(len(data_test))
X = pd.concat([data_temp.loc[:, features],data_test.loc[:, features]])
Y = pd.concat([data_temp[target],data_test[target]])
X.index=range(len(X))
Y.index=range(len(Y))
X_max = X[:-test_num].max()
X_min = X[:-test_num].min()
X_s = (X-X_min)/(X_max-X_min)

train_num = len(X) - valid_num - test_num - window_size_LR + 1
temp_X = X_s.copy()
temp_Y = Y.copy()

temp_X_w, temp_Y_w = make_dataset_D(temp_X, temp_Y, window_size_LR)
temp_X_w_1 = temp_X_w.reshape(temp_X_w.shape[0],-1)

X_train = temp_X_w_1[:train_num]
X_valid = temp_X_w_1[train_num:train_num+valid_num]
X_test = temp_X_w_1[-test_num:]

y_train = temp_Y_w[:train_num]
y_valid = temp_Y_w[train_num:train_num+valid_num]
y_test = temp_Y_w[-test_num:]

X_train_total = np.vstack([X_train, X_valid])
y_train_total = np.concatenate([y_train, y_valid])

test_fold = np.array([-1]*len(X_train) + [0]*len(X_valid))
ps = PredefinedSplit(test_fold)

rmse_scorer = make_scorer(
        lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),  # ← squared 대신 직접 루트
        greater_is_better=False
    )

In [28]:
n_boot = 1000

In [27]:
for b_num in range(n_boot):
    print('LR, b_num =',b_num)
    rng = np.random.default_rng(42 + b_num)
    idx = block_bootstrap_indices(len(y_train_total), 13, rng)
    
    X_boot_total = X_train_total[idx]
    y_boot_total = y_train_total[idx]
    
    LR_model = LinearRegression()
    LR_model.fit(X_boot_total, y_boot_total)
    
    LR_pred_train = LR_model.predict(X_train_total)
    LR_pred_test = LR_model.predict(X_test)
    
    LR_pred_train = np.maximum(0, LR_pred_train)
    LR_pred_test = np.maximum(0, LR_pred_test)

    joblib.dump(LR_model, './model_CI_best/LR_model_Regression_'+str(b_num)+'.pkl')

LR, b_num = 0
LR, b_num = 1
LR, b_num = 2
LR, b_num = 3
LR, b_num = 4
LR, b_num = 5
LR, b_num = 6
LR, b_num = 7
LR, b_num = 8
LR, b_num = 9
LR, b_num = 10
LR, b_num = 11
LR, b_num = 12
LR, b_num = 13
LR, b_num = 14
LR, b_num = 15
LR, b_num = 16
LR, b_num = 17
LR, b_num = 18
LR, b_num = 19
LR, b_num = 20
LR, b_num = 21
LR, b_num = 22
LR, b_num = 23
LR, b_num = 24
LR, b_num = 25
LR, b_num = 26
LR, b_num = 27
LR, b_num = 28
LR, b_num = 29
LR, b_num = 30
LR, b_num = 31
LR, b_num = 32
LR, b_num = 33
LR, b_num = 34
LR, b_num = 35
LR, b_num = 36
LR, b_num = 37
LR, b_num = 38
LR, b_num = 39
LR, b_num = 40
LR, b_num = 41
LR, b_num = 42
LR, b_num = 43
LR, b_num = 44
LR, b_num = 45
LR, b_num = 46
LR, b_num = 47
LR, b_num = 48
LR, b_num = 49
LR, b_num = 50
LR, b_num = 51
LR, b_num = 52
LR, b_num = 53
LR, b_num = 54
LR, b_num = 55
LR, b_num = 56
LR, b_num = 57
LR, b_num = 58
LR, b_num = 59
LR, b_num = 60
LR, b_num = 61
LR, b_num = 62
LR, b_num = 63
LR, b_num = 64
LR, b_num = 65
LR, b_num = 66
LR, b

In [29]:
window_size_RF = 7

In [30]:
data_temp = data_train[(data_train['Year']>start_year) | (data_train['Week']>53-window_size_RF+1)]
data_temp.index = range(len(data_temp))
data_test.index = range(len(data_test))
X = pd.concat([data_temp.loc[:, features],data_test.loc[:, features]])
Y = pd.concat([data_temp[target],data_test[target]])
X.index=range(len(X))
Y.index=range(len(Y))
X_max = X[:-test_num].max()
X_min = X[:-test_num].min()
X_s = (X-X_min)/(X_max-X_min)

train_num = len(X) - valid_num - test_num - window_size_RF + 1
temp_X = X_s.copy()
temp_Y = Y.copy()

temp_X_w, temp_Y_w = make_dataset_D(temp_X, temp_Y, window_size_RF)
temp_X_w_1 = temp_X_w.reshape(temp_X_w.shape[0],-1)

X_train = temp_X_w_1[:train_num]
X_valid = temp_X_w_1[train_num:train_num+valid_num]
X_test = temp_X_w_1[-test_num:]

y_train = temp_Y_w[:train_num]
y_valid = temp_Y_w[train_num:train_num+valid_num]
y_test = temp_Y_w[-test_num:]

X_train_total = np.vstack([X_train, X_valid])
y_train_total = np.concatenate([y_train, y_valid])

test_fold = np.array([-1]*len(X_train) + [0]*len(X_valid))
ps = PredefinedSplit(test_fold)

rmse_scorer = make_scorer(
        lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),  # ← squared 대신 직접 루트
        greater_is_better=False
    )

In [31]:
random_forest_model = RandomForestRegressor(random_state=seed_value)
# param_grid_rf = {
#     'n_estimators': (100, 500),  # 트리 개수 범위
#     'max_features': ['sqrt', 'log2'],  # 특성 선택 기준
#     'max_depth': (30, 70),  # 트리 최대 깊이 범위
#     'min_samples_split': (5, 15),  # 노드 분할 기준 범위
#     'min_samples_leaf': (2, 6),  # 리프 노드 최소 샘플 수 범위
#     'bootstrap': [True]  # 부트스트랩 샘플링 유지
# }
param_grid_rf = {
    'n_estimators': (100, 1000),  # 트리 개수 범위
    'max_features': ['sqrt', 'log2'],  # 특성 선택 기준
    'max_depth': (2, 70),  # 트리 최대 깊이 범위
    'min_samples_split': (3, 15),  # 노드 분할 기준 범위
    'min_samples_leaf': (2, 10),  # 리프 노드 최소 샘플 수 범위
    'bootstrap': [True]# 부트스트랩 샘플링 유지
}

Bays_rf = BayesSearchCV(
    estimator=random_forest_model,
    search_spaces=param_grid_rf,
    scoring='neg_mean_squared_error',  # MSE 사용
    cv=5,                # 5-fold 교차 검증
    n_iter=100,          # 최대 100번의 탐색
    random_state=seed_value,
    verbose=1,           # 탐색 진행 상황 출력
    n_jobs=-1            # 병렬 처리
)

Bays_rf.fit(X_train_total, y_train_total)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi

AttributeError: 'bool' object has no attribute 'all'

AttributeError: 'bool' object has no attribute 'all'

BayesSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
              n_iter=100, n_jobs=-1, random_state=42,
              scoring='neg_mean_squared_error',
              search_spaces={'bootstrap': [True], 'max_depth': (2, 70),
                             'max_features': ['sqrt', 'log2'],
                             'min_samples_leaf': (2, 10),
                             'min_samples_split': (3, 15),
                             'n_estimators': (100, 1000)},
              verbose=1)

In [32]:
Bays_rf.best_params_

OrderedDict([('bootstrap', True),
             ('max_depth', 70),
             ('max_features', 'log2'),
             ('min_samples_leaf', 7),
             ('min_samples_split', 8),
             ('n_estimators', 100)])

In [32]:
for b_num in range(n_boot):
    print('RF, b_num =',b_num)
    rng = np.random.default_rng(42 + b_num)
    idx = block_bootstrap_indices(len(y_train_total), 13, rng)
    
    X_boot_total = X_train_total[idx]
    y_boot_total = y_train_total[idx]
    
    RF_model = RandomForestRegressor(random_state=seed_value, **Bays_rf.best_params_)
    RF_model.fit(X_boot_total, y_boot_total)
    RF_pred_train = RF_model.predict(X_train_total)
    RF_pred_test = RF_model.predict(X_test)

    joblib.dump(RF_model, './model_CI_best/RF_model_Regression_'+str(b_num)+'.pkl')

RF, b_num = 0
RF, b_num = 1
RF, b_num = 2
RF, b_num = 3
RF, b_num = 4
RF, b_num = 5
RF, b_num = 6
RF, b_num = 7
RF, b_num = 8
RF, b_num = 9
RF, b_num = 10
RF, b_num = 11
RF, b_num = 12
RF, b_num = 13
RF, b_num = 14
RF, b_num = 15
RF, b_num = 16
RF, b_num = 17
RF, b_num = 18
RF, b_num = 19
RF, b_num = 20
RF, b_num = 21
RF, b_num = 22
RF, b_num = 23
RF, b_num = 24
RF, b_num = 25
RF, b_num = 26
RF, b_num = 27
RF, b_num = 28
RF, b_num = 29
RF, b_num = 30
RF, b_num = 31
RF, b_num = 32
RF, b_num = 33
RF, b_num = 34
RF, b_num = 35
RF, b_num = 36
RF, b_num = 37
RF, b_num = 38
RF, b_num = 39
RF, b_num = 40
RF, b_num = 41
RF, b_num = 42
RF, b_num = 43
RF, b_num = 44
RF, b_num = 45
RF, b_num = 46
RF, b_num = 47
RF, b_num = 48
RF, b_num = 49
RF, b_num = 50
RF, b_num = 51
RF, b_num = 52
RF, b_num = 53
RF, b_num = 54
RF, b_num = 55
RF, b_num = 56
RF, b_num = 57
RF, b_num = 58
RF, b_num = 59
RF, b_num = 60
RF, b_num = 61
RF, b_num = 62
RF, b_num = 63
RF, b_num = 64
RF, b_num = 65
RF, b_num = 66
RF, b

In [33]:
window_size_XGB = 8

In [34]:
data_temp = data_train[(data_train['Year']>start_year) | (data_train['Week']>53-window_size_XGB+1)]
data_temp.index = range(len(data_temp))
data_test.index = range(len(data_test))
X = pd.concat([data_temp.loc[:, features],data_test.loc[:, features]])
Y = pd.concat([data_temp[target],data_test[target]])
X.index=range(len(X))
Y.index=range(len(Y))
X_max = X[:-test_num].max()
X_min = X[:-test_num].min()
X_s = (X-X_min)/(X_max-X_min)

train_num = len(X) - valid_num - test_num - window_size_XGB + 1
temp_X = X_s.copy()
temp_Y = Y.copy()

temp_X_w, temp_Y_w = make_dataset_D(temp_X, temp_Y, window_size_XGB)
temp_X_w_1 = temp_X_w.reshape(temp_X_w.shape[0],-1)

X_train = temp_X_w_1[:train_num]
X_valid = temp_X_w_1[train_num:train_num+valid_num]
X_test = temp_X_w_1[-test_num:]

y_train = temp_Y_w[:train_num]
y_valid = temp_Y_w[train_num:train_num+valid_num]
y_test = temp_Y_w[-test_num:]

X_train_total = np.vstack([X_train, X_valid])
y_train_total = np.concatenate([y_train, y_valid])

test_fold = np.array([-1]*len(X_train) + [0]*len(X_valid))
ps = PredefinedSplit(test_fold)

rmse_scorer = make_scorer(
        lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),  # ← squared 대신 직접 루트
        greater_is_better=False
    )

In [35]:
np.random.seed(seed_value)
random.seed(seed_value)
# XGBoost 모델 생성
XGB_model = XGBRegressor(
    base_score=float(np.mean(y_train_total)),  # 타깃 평균으로 초기값 보정
    random_state=seed_value
)

# 베이지안 최적화를 위한 하이퍼파라미터 범위 설정 (과적합 방지 적용)
XGB_params = {
    'learning_rate': (0.01, 0.2, 'log-uniform'),  # 학습률 (최적화 안정성 고려)
    'n_estimators': (100, 1000),                    # 추정기 수 (과적합 방지)
    'max_depth': (3, 10),                          # 최대 깊이 (깊이 제한)
    'colsample_bytree': (0.3, 0.9),               # 열 샘플링 비율 (과적합 방지)
    'subsample': (0.4, 0.9)                       # 데이터 샘플링 비율 (일반화 향상)
}


# 베이지안 최적화 객체 생성
Bays_xgb = BayesSearchCV(
    estimator=XGB_model,
    search_spaces=XGB_params,
    scoring=rmse_scorer,  # 사용자 정의 MSE 스코어러
    cv=ps,                # 5-fold 교차 검증
    n_iter=100,          # 최대 100번의 탐색
    random_state=seed_value,
    verbose=0            # 탐색 진행 상황 출력
)
Bays_xgb.fit(X_train_total, y_train_total)

,estimator,"XGBRegressor(...ree=None, ...)"
,search_spaces,"{'colsample_bytree': (0.3, ...), 'learning_rate': (0.01, ...), 'max_depth': (3, ...), 'n_estimators': (100, ...), ...}"
,optimizer_kwargs,None
,n_iter,100
,scoring,make_scorer(<...hod='predict')
,fit_params,None
,n_jobs,1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,"PredefinedSpl......, 0, 0]))"


In [36]:
Bays_xgb.best_params_

OrderedDict([('colsample_bytree', 0.3511230548379923),
             ('learning_rate', 0.01),
             ('max_depth', 9),
             ('n_estimators', 176),
             ('subsample', 0.4)])

In [37]:
for b_num in range(n_boot):
    print('XGB, b_num =',b_num)
    rng = np.random.default_rng(42 + b_num)
    idx = block_bootstrap_indices(len(y_train_total), 13, rng)
    
    X_boot_total = X_train_total[idx]
    y_boot_total = y_train_total[idx]
    
    XGB_model = XGBRegressor(base_score=float(np.mean(y_train_total)),
                             learning_rate=Bays_xgb.best_params_['learning_rate'],
                          n_estimators=Bays_xgb.best_params_['n_estimators'], 
                          max_depth=Bays_xgb.best_params_['max_depth'],
                          colsample_bytree=Bays_xgb.best_params_['colsample_bytree'],
                          subsample=Bays_xgb.best_params_['subsample'],
                          random_state=42)
    XGB_model.fit(X_boot_total, y_boot_total)
    XGB_pred_train = XGB_model.predict(X_train_total)
    XGB_pred_test = XGB_model.predict(X_test)
    
    XGB_model.save_model(f'./model_CI_best/XGB_model_Regression_{b_num}.json')
    # joblib.dump(XGB_model, './model_CI_best/XGB_model_Regression_'+str(b_num)+'.pkl')

XGB, b_num = 0
XGB, b_num = 1
XGB, b_num = 2
XGB, b_num = 3
XGB, b_num = 4
XGB, b_num = 5
XGB, b_num = 6
XGB, b_num = 7
XGB, b_num = 8
XGB, b_num = 9
XGB, b_num = 10
XGB, b_num = 11
XGB, b_num = 12
XGB, b_num = 13
XGB, b_num = 14
XGB, b_num = 15
XGB, b_num = 16
XGB, b_num = 17
XGB, b_num = 18
XGB, b_num = 19
XGB, b_num = 20
XGB, b_num = 21
XGB, b_num = 22
XGB, b_num = 23
XGB, b_num = 24
XGB, b_num = 25
XGB, b_num = 26
XGB, b_num = 27
XGB, b_num = 28
XGB, b_num = 29
XGB, b_num = 30
XGB, b_num = 31
XGB, b_num = 32
XGB, b_num = 33
XGB, b_num = 34
XGB, b_num = 35
XGB, b_num = 36
XGB, b_num = 37
XGB, b_num = 38
XGB, b_num = 39
XGB, b_num = 40
XGB, b_num = 41
XGB, b_num = 42
XGB, b_num = 43
XGB, b_num = 44
XGB, b_num = 45
XGB, b_num = 46
XGB, b_num = 47
XGB, b_num = 48
XGB, b_num = 49
XGB, b_num = 50
XGB, b_num = 51
XGB, b_num = 52
XGB, b_num = 53
XGB, b_num = 54
XGB, b_num = 55
XGB, b_num = 56
XGB, b_num = 57
XGB, b_num = 58
XGB, b_num = 59
XGB, b_num = 60
XGB, b_num = 61
XGB, b_num = 62
XG

In [27]:
window_size_DNN = 7

In [28]:
class DNN(nn.Module):
    def __init__(self, hidden_dim=128, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(len(features)*window_size_DNN, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

In [29]:
data_temp = data_train[(data_train['Year']>start_year) | (data_train['Week']>53-window_size_DNN+1)]
data_temp.index = range(len(data_temp))
data_test.index = range(len(data_test))
X = pd.concat([data_temp.loc[:, features],data_test.loc[:, features]])
Y = pd.concat([data_temp[target],data_test[target]])
X.index=range(len(X))
Y.index=range(len(Y))
X_max = X[:-test_num].max()
X_min = X[:-test_num].min()
X_s = (X-X_min)/(X_max-X_min)

train_num = len(X) - valid_num - test_num - window_size_DNN + 1
temp_X = X_s.copy()
temp_Y = Y.copy()

temp_X_w, temp_Y_w = make_dataset_D(temp_X, temp_Y, window_size_DNN)
temp_X_w_1 = temp_X_w.reshape(temp_X_w.shape[0],-1)

X_train = temp_X_w_1[:train_num]
X_valid = temp_X_w_1[train_num:train_num+valid_num]
X_test = temp_X_w_1[-test_num:]

y_train = temp_Y_w[:train_num]
y_valid = temp_Y_w[train_num:train_num+valid_num]
y_test = temp_Y_w[-test_num:]

X_train_total = np.vstack([X_train, X_valid])
y_train_total = np.concatenate([y_train, y_valid])

In [30]:
X_train_D = torch.Tensor(X_train)
X_valid_D = torch.Tensor(X_valid)
X_test_D = torch.Tensor(X_test)

y_train_D = torch.Tensor(y_train).reshape(-1,1)
y_valid_D = torch.Tensor(y_valid).reshape(-1,1)
y_test_D = torch.Tensor(y_test).reshape(-1,1)

X_train_total_D = torch.cat([X_train_D, X_valid_D], dim=0)
y_train_total_D = torch.cat([y_train_D, y_valid_D], dim=0)

In [32]:
for b_num in range(877,n_boot):
    print('DNN, b_num =',b_num)
    rng = np.random.default_rng(42 + b_num)
    idx = block_bootstrap_indices(len(y_train_total_D), 13, rng)
    
    X_boot_total_D = X_train_total_D[idx]
    y_boot_total_D = y_train_total_D[idx]
    
    train_dataset_total_D = TensorDataset(X_boot_total_D, y_boot_total_D)
    train_loader_total_D = DataLoader(train_dataset_total_D, batch_size=16, pin_memory=True)
    
    loss_train_record=[]
    loss_valid_record=[]

    set_seed(42)
    model = DNN(hidden_dim=128, dropout=0.5).to(device)
    
    criterion = nn.MSELoss()  # 평균 제곱 오차
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)  # 확률적 경사 하강법
    
    best_idx = 256
    
    for epoch in range(best_idx+1):
        model.train()
        for inputs, targets in train_loader_total_D:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
    
            # 역전파 및 최적화
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
              
    model.eval()
    with torch.no_grad():
        DNN_pred_train = model(X_train_total_D.to(device))
        loss_train = criterion(DNN_pred_train,y_train_total_D.to(device))
        DNN_pred_test = model(X_test_D.to(device))
        loss_test = criterion(DNN_pred_test,y_test_D.to(device))
        
    torch.save(model.state_dict(), f'./model_CI_best/DNN_model_Regression_{b_num}.pth')

DNN, b_num = 877
DNN, b_num = 878
DNN, b_num = 879
DNN, b_num = 880
DNN, b_num = 881
DNN, b_num = 882
DNN, b_num = 883
DNN, b_num = 884
DNN, b_num = 885
DNN, b_num = 886
DNN, b_num = 887
DNN, b_num = 888
DNN, b_num = 889
DNN, b_num = 890
DNN, b_num = 891
DNN, b_num = 892
DNN, b_num = 893
DNN, b_num = 894
DNN, b_num = 895
DNN, b_num = 896
DNN, b_num = 897
DNN, b_num = 898
DNN, b_num = 899
DNN, b_num = 900
DNN, b_num = 901
DNN, b_num = 902
DNN, b_num = 903
DNN, b_num = 904
DNN, b_num = 905
DNN, b_num = 906
DNN, b_num = 907
DNN, b_num = 908
DNN, b_num = 909
DNN, b_num = 910
DNN, b_num = 911
DNN, b_num = 912
DNN, b_num = 913
DNN, b_num = 914
DNN, b_num = 915
DNN, b_num = 916
DNN, b_num = 917
DNN, b_num = 918
DNN, b_num = 919
DNN, b_num = 920
DNN, b_num = 921
DNN, b_num = 922
DNN, b_num = 923
DNN, b_num = 924
DNN, b_num = 925
DNN, b_num = 926
DNN, b_num = 927
DNN, b_num = 928
DNN, b_num = 929
DNN, b_num = 930
DNN, b_num = 931
DNN, b_num = 932
DNN, b_num = 933
DNN, b_num = 934
DNN, b_num = 9

In [58]:
window_size_LSTM = 8

In [59]:
class LSTM(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=3,
                 dropout=0.4):
        super(LSTM, self).__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=len(features),
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.fc1 = nn.Linear(hidden_dim, hidden_dim//2)
        self.fc2 = nn.Linear(hidden_dim//2, 1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        h0 = torch.zeros(
            self.num_layers,
            x.shape[0],
            self.hidden_dim,
            device=x.device
        )

        c0 = torch.zeros(
            self.num_layers,
            x.shape[0],
            self.hidden_dim,
            device=x.device
        )

        x, _ = self.lstm(x, (h0, c0))

        x = x[:, -1, :]

        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)

        return x

In [60]:
data_temp = data_train[(data_train['Year']>start_year) | (data_train['Week']>53-window_size_LSTM+1)]
data_temp.index = range(len(data_temp))
data_test.index = range(len(data_test))
X = pd.concat([data_temp.loc[:, features],data_test.loc[:, features]])
Y = pd.concat([data_temp[target],data_test[target]])
X.index=range(len(X))
Y.index=range(len(Y))
X_max = X[:-test_num].max()
X_min = X[:-test_num].min()
X_s = (X-X_min)/(X_max-X_min)

train_num = len(X) - valid_num - test_num - window_size_LSTM + 1
temp_X = X_s.copy()
temp_Y = Y.copy()

In [61]:
X_w, Y_w = make_dataset_D(X_s, Y, window_size_LSTM)

X_train_L = torch.Tensor(X_w[:train_num])
X_valid_L = torch.Tensor(X_w[train_num:train_num+valid_num])
X_test_L = torch.Tensor(X_w[-test_num:])

y_train_L = torch.Tensor(Y_w[:train_num]).reshape(-1,1)
y_valid_L = torch.Tensor(Y_w[train_num:train_num+valid_num]).reshape(-1,1)
y_test_L = torch.Tensor(Y_w[-test_num:]).reshape(-1,1)

X_train_total_L = torch.cat([X_train_L, X_valid_L], dim=0)
y_train_total_L = torch.cat([y_train_L, y_valid_L], dim=0)

In [63]:
for b_num in range(800,n_boot):
    print('LSTM, b_num =',b_num)
    rng = np.random.default_rng(42 + b_num)
    idx = block_bootstrap_indices(len(y_train_total_L), 13, rng)
    
    X_boot_total_L = X_train_total_L[idx]
    y_boot_total_L = y_train_total_L[idx]
    
    train_dataset_total_L = TensorDataset(X_boot_total_L, y_boot_total_L)
    train_loader_total_L = DataLoader(train_dataset_total_L, batch_size=32, pin_memory=True)
    
    loss_train_record=[]
    loss_valid_record=[]

    set_seed(42)
    model = LSTM(hidden_dim=64, num_layers=3, dropout=0.4).to(device)
    
    criterion = nn.MSELoss()  # 평균 제곱 오차
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0002)  # 확률적 경사 하강법
    
    best_idx = 962
    
    for epoch in range(best_idx+1):
        model.train()
        for inputs, targets in train_loader_total_L:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
    
            # 역전파 및 최적화
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
              
    model.eval()
    with torch.no_grad():
        LSTM_pred_train = model(X_train_total_L.to(device))
        loss_train = criterion(LSTM_pred_train, y_train_total_L.to(device))
    
        LSTM_pred_test = model(X_test_L.to(device))
        loss_test = criterion(LSTM_pred_test, y_test_L.to(device))
        
    torch.save(model.state_dict(), f'./model_CI_best/LSTM_model_Regression_{b_num}.pth')

LSTM, b_num = 800
LSTM, b_num = 801
LSTM, b_num = 802
LSTM, b_num = 803
LSTM, b_num = 804
LSTM, b_num = 805
LSTM, b_num = 806
LSTM, b_num = 807
LSTM, b_num = 808
LSTM, b_num = 809
LSTM, b_num = 810
LSTM, b_num = 811
LSTM, b_num = 812
LSTM, b_num = 813
LSTM, b_num = 814
LSTM, b_num = 815
LSTM, b_num = 816
LSTM, b_num = 817
LSTM, b_num = 818
LSTM, b_num = 819
LSTM, b_num = 820
LSTM, b_num = 821
LSTM, b_num = 822
LSTM, b_num = 823
LSTM, b_num = 824
LSTM, b_num = 825
LSTM, b_num = 826
LSTM, b_num = 827
LSTM, b_num = 828
LSTM, b_num = 829
LSTM, b_num = 830
LSTM, b_num = 831
LSTM, b_num = 832
LSTM, b_num = 833
LSTM, b_num = 834
LSTM, b_num = 835
LSTM, b_num = 836
LSTM, b_num = 837
LSTM, b_num = 838
LSTM, b_num = 839
LSTM, b_num = 840
LSTM, b_num = 841
LSTM, b_num = 842
LSTM, b_num = 843
LSTM, b_num = 844
LSTM, b_num = 845
LSTM, b_num = 846
LSTM, b_num = 847
LSTM, b_num = 848
LSTM, b_num = 849
LSTM, b_num = 850
LSTM, b_num = 851
LSTM, b_num = 852
LSTM, b_num = 853
LSTM, b_num = 854
LSTM, b_nu